# Темы 12-14: Кластеризация (ОТВЕТЫ)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

df = pd.merge(pd.read_csv('data/transactions.csv'), pd.read_csv('data/customers.csv'), on='customer_id')
df = pd.merge(df, pd.read_csv('data/products.csv'), on='product_id')
df['total_sum'] = df['price'] * df['quantity']
df['date'] = pd.to_datetime(df['date'])

### Задание 1

In [ ]:
max_date = df['date'].max()
rfm = df.groupby('customer_id').agg(
    recency=('date', lambda x: (max_date - x.max()).days),
    frequency=('transaction_id', 'count'),
    monetary=('total_sum', 'sum')
).reset_index()
rfm.head()

### Задание 2

In [ ]:
rfm[['recency','frequency','monetary']].describe()

### Задание 3

In [ ]:
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['recency','frequency','monetary']])

### Задание 4

In [ ]:
inertias = []
K = range(2, 11)
for k in K:
    km = KMeans(n_clusters=k, random_state=42, n_init='auto')
    km.fit(rfm_scaled)
    inertias.append(km.inertia_)
plt.figure(figsize=(8,5))
plt.plot(K, inertias, 'bo-')
plt.xlabel('k')
plt.ylabel('Inertia')
plt.title('Метод локтя')
plt.show()

### Задание 5

In [ ]:
k_opt = 4
kmeans = KMeans(n_clusters=k_opt, random_state=42, n_init='auto')
rfm['cluster'] = kmeans.fit_predict(rfm_scaled)
rfm['cluster'].value_counts()

### Задание 6

In [ ]:
print(rfm.groupby('cluster')[['recency','frequency','monetary']].mean())

### Задание 7

In [ ]:
plt.figure(figsize=(10,6))
sns.scatterplot(data=rfm, x='frequency', y='monetary', hue='cluster', palette='viridis', alpha=0.6)
plt.title('Сегменты клиентов')
plt.show()

### Задание 8

In [ ]:
from mpl_toolkits.mplot3d import Axes3D
fig = plt.figure(figsize=(10,8))
ax = fig.add_subplot(111, projection='3d')
scatter = ax.scatter(rfm['recency'], rfm['frequency'], rfm['monetary'], c=rfm['cluster'], cmap='viridis')
ax.set_xlabel('Recency')
ax.set_ylabel('Frequency')
ax.set_zlabel('Monetary')
plt.colorbar(scatter)
plt.show()

### Задание 9

In [ ]:
means = rfm.groupby('cluster')[['recency','frequency','monetary']].mean()
# Назначаем имена на основе RFM-средних
segment_map = {}
for c in means.index:
    r, f, m = means.loc[c]
    if f > means['frequency'].median() and m > means['monetary'].median():
        segment_map[c] = 'VIP'
    elif r > means['recency'].median():
        segment_map[c] = 'Спящие'
    elif f <= means['frequency'].median():
        segment_map[c] = 'Новички'
    else:
        segment_map[c] = 'Обычные'
rfm['segment'] = rfm['cluster'].map(segment_map)
print(rfm['segment'].value_counts())

### Задание 10

In [ ]:
rfm['segment'].value_counts().plot.pie(autopct='%1.1f%%', figsize=(7,7))
plt.title('Размер сегментов')
plt.ylabel('')
plt.show()

### Задание 11

In [ ]:
from sklearn.metrics import silhouette_score
score = silhouette_score(rfm_scaled, rfm['cluster'])
print(f'Silhouette Score: {score:.4f}')